In [ ]:
# load data fitur hasil ekstraksi dari preprocessing.ipynb

import os
import json
import time
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report, confusion_matrix,
                             ConfusionMatrixDisplay)

SEED = 42   # harus sama dengan preprocessing.ipynb dan mlp.ipynb
os.makedirs("results", exist_ok=True)

data = np.load("data/features.npz")
X_train, y_train = data["X_train"], data["y_train"]
X_test,  y_test  = data["X_test"],  data["y_test"]
class_names = data["class_names"]

print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Jumlah kelas:", len(class_names))

In [ ]:
# training model Random Forest

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    max_features="sqrt",
    min_samples_leaf=1,
    n_jobs=-1,
    random_state=SEED,
)

start = time.time()
rf.fit(X_train, y_train)
train_time = time.time() - start

print(f"Waktu training : {train_time:.1f} detik")
print("Jumlah pohon   :", rf.n_estimators)

In [ ]:
# evaluasi pada data testing

y_pred = rf.predict(X_test)

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average="macro", zero_division=0)
rec  = recall_score(y_test, y_pred, average="macro", zero_division=0)
f1   = f1_score(y_test, y_pred, average="macro", zero_division=0)

print(f"Accuracy  : {acc:.4f}")
print(f"Precision : {prec:.4f}")
print(f"Recall    : {rec:.4f}")
print(f"F1-score  : {f1:.4f}")

print("\nLaporan per kelas:")
print(classification_report(y_test, y_pred, target_names=class_names, zero_division=0))

# Akurasi train sebagai pembanding untuk melihat overfitting
train_acc = accuracy_score(y_train, rf.predict(X_train))
print(f"Akurasi train: {train_acc:.4f} | Akurasi test: {acc:.4f}")

In [ ]:
# visualisasi confusion matrix

cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(10, 9))
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(
    ax=ax, cmap="Greens", xticks_rotation=90, colorbar=False
)
ax.set_title("Confusion Matrix Random Forest")
plt.tight_layout()
plt.savefig("results/figures/rf_confusion_matrix.png", dpi=150)
plt.show()

In [ ]:
# save hasil evaluasi ke file JSON dan prediksi ke file NPY

hasil = {
    "model": "Random Forest",
    "accuracy": float(acc),
    "precision_macro": float(prec),
    "recall_macro": float(rec),
    "f1_macro": float(f1),
    "train_accuracy": float(train_acc),
    "train_time_sec": float(train_time),
    "params": {
        "n_estimators": 300,
        "max_depth": None,
        "max_features": "sqrt",
        "min_samples_leaf": 1,
        "pca": None,
    },
}

with open("results/rf_metrics.json", "w") as f:
    json.dump(hasil, f, indent=2)

np.save("results/rf_y_pred.npy", y_pred)
print("Tersimpan di results/rf_metrics.json")